# 3901161_1386_EasyTSLite.csv

This CSV is an Argo profile file with a metadata comment block at the top and tabular measurements below.

## File Structure

The first lines begin with `#`. These are comments used as metadata, not data rows. They describe the profile and the source of the file.

In this file, the metadata includes:

- format: `EasyOneArgoTSLite`
- creation date: `2026-03-16T04:30:37Z`
- creation centre: `Ifremer`
- data source DOI: `https://doi.org/10.17882/42182#126945`
- platform number: `3901161`
- cycle number: `1386`
- profile date: `2017-02-09T23:29:42Z`
- profile latitude: `6.144`
- profile longitude: `-119.519`

The comment lines also define the variables used in the profile:

- pressure = sea water pressure, measured in decibar
- temperature = sea temperature in-situ on the ITS-90 scale
- salinity = practical salinity

After the comments, the file switches to a comma-separated header row:

- pressure (decibar)
- temperature (degree_celsius)
- salinity (dimensionless)
- pressure_error (decibar)
- temperature_error (degree_celsius)
- salinity_error (dimensionless)

Each following row is one measurement at a given pressure level.

## How To Read It In Pandas

Because the metadata lines start with `#`, pandas should skip them when reading the file:

```python
single_argo = pd.read_csv('3901161_1386_EasyTSLite.csv', comment='#')
```

That keeps the header row and measurement rows while ignoring the comment block.

## Notes

- The parser error happens if pandas tries to interpret the comment block as tabular data.
- The file is comma-separated, so the default delimiter is correct once the comment lines are skipped.

In [6]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re


csv_path = Path('3901161_1386_EasyTSLite.csv')
argo_metadata = {}
with csv_path.open('r', encoding='utf-8', errors='replace') as csv_file:
    for line in csv_file:
        if not line.startswith('#'):
            break
        comment = line[1:].strip()
        if '=' in comment:
            key, value = comment.split('=', 1)
            argo_metadata[key.strip()] = value.strip()
        else:
            parts = comment.split(None, 1)
            if len(parts) == 2:
                argo_metadata[parts[0].strip()] = parts[1].strip()
            else:
                argo_metadata[comment] = None

single_argo = pd.read_csv(csv_path, comment='#')

# Add one constant metadata column per key for every row in the profile table.
for key, value in argo_metadata.items():
    safe_key = re.sub(r'[^0-9a-zA-Z_]+', '_', key.strip().lower()).strip('_')
    meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
    single_argo[meta_col] = value

single_argo.attrs['metadata'] = argo_metadata
single_argo.head()

,pressure (decibar),temperature (degree_celsius),salinity (dimensionless),pressure_error (decibar),temperature_error (degree_celsius),salinity_error (dimensionless),meta_format,meta_creation_date,meta_creation_centre,meta_creation_centre_pid,...,meta_platform_number,meta_cycle_number,meta_direction_of_profile,meta_data_mode,meta_profile_date,meta_profile_latitude,meta_profile_longitude,meta_pressure,meta_temperature,meta_salinity
0,2.0,27.643,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
1,5.0,27.636,34.077,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
2,10.0,27.528,34.076,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
3,15.0,27.435,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity
4,20.0,27.424,34.071,2.42,0.003,0.01,EasyOneArgoTSLite,2026-03-16T04:30:37Z,Ifremer,https://ror.org/044jxhp58,...,3901161,1386,A,D,2017-02-09T23:29:42Z,6.144,-119.519,sea water pressure equals 0 at sea-level,sea temperature in-situ ITS-90 scale,practical salinity


## Multi-file Argo Ingestion Setup

Set the folder placeholder in the next cell to the directory that contains your Argo CSV files.

Use one of these modes:

- `run_in_memory(...)` for smaller datasets that fit in RAM.
- `run_streaming_to_parquet(...)` for large datasets (recommended for very large folders).

The code searches recursively with `**/*.csv`, so nested folders are supported.

In [ ]:
import pandas as pd
import re
from pathlib import Path


def parse_argo_csv(csv_path: Path) -> pd.DataFrame:
    """Parse one Argo CSV file, preserving # metadata as DataFrame columns."""
    csv_path = Path(csv_path)
    argo_metadata = {}

    with csv_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("#"):
                break
            comment = line[1:].strip()
            if "=" in comment:
                key, value = comment.split("=", 1)
                argo_metadata[key.strip()] = value.strip()
            else:
                parts = comment.split(None, 1)
                if len(parts) == 2:
                    argo_metadata[parts[0].strip()] = parts[1].strip()
                else:
                    argo_metadata[comment] = None

    df = pd.read_csv(csv_path, comment="#")

    for key, value in argo_metadata.items():
        safe_key = re.sub(r"[^0-9a-zA-Z_]+", "_", key.strip().lower()).strip("_")
        meta_col = f"meta_{safe_key}" if safe_key else "meta_unknown"
        df[meta_col] = value

    df["source_file"] = csv_path.name
    return df


# =========================
# PLACEHOLDER: SET THIS
# =========================
CSV_ROOT = Path("/ABSOLUTE/PATH/TO/EasyOneArgoTSLite_20260316T043037Z/data")
CSV_PATTERN = "**/*.csv"
OUTPUT_PARQUET = Path("argo_combined.parquet")


def list_csv_files(csv_root: Path, pattern: str = "**/*.csv") -> list[Path]:
    if not csv_root.exists():
        raise FileNotFoundError(f"CSV_ROOT not found: {csv_root}")
    files = sorted(csv_root.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No CSV files found under: {csv_root} with pattern: {pattern}")
    return files


def run_in_memory(csv_root: Path, pattern: str = "**/*.csv", output_path: Path = OUTPUT_PARQUET):
    csv_files = list_csv_files(csv_root, pattern)
    all_dfs = []
    failed = []

    for i, csv_path in enumerate(csv_files, start=1):
        try:
            df = parse_argo_csv(csv_path)
            all_dfs.append(df)
        except Exception as e:
            print(f"[{i}/{len(csv_files)}] Failed: {csv_path} | {e}")
            failed.append(str(csv_path))

    if not all_dfs:
        raise RuntimeError("No files were parsed successfully.")

    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_parquet(output_path, index=False)

    print(f"Total rows: {len(combined):,} | Files: {len(all_dfs)} | Failed: {len(failed)}")
    print(f"Saved: {output_path.resolve()}")
    return combined, failed


def run_streaming_to_parquet(csv_root: Path, pattern: str = "**/*.csv", output_path: Path = OUTPUT_PARQUET):
    import pyarrow as pa
    import pyarrow.parquet as pq

    csv_files = list_csv_files(csv_root, pattern)
    writer = None
    failed = []
    ok = 0

    try:
        for i, csv_path in enumerate(csv_files, start=1):
            try:
                df = parse_argo_csv(csv_path)
                table = pa.Table.from_pandas(df)

                if writer is None:
                    writer = pq.ParquetWriter(str(output_path), table.schema)
                writer.write_table(table)
                ok += 1

            except Exception as e:
                print(f"[{i}/{len(csv_files)}] Failed: {csv_path} | {e}")
                failed.append(str(csv_path))
    finally:
        if writer is not None:
            writer.close()

    if ok == 0:
        raise RuntimeError("No files were written to parquet.")

    print(f"Files written: {ok} | Failed: {len(failed)}")
    print(f"Saved: {output_path.resolve()}")
    return failed


# Example usage (pick one):
# combined, failed = run_in_memory(CSV_ROOT, CSV_PATTERN, OUTPUT_PARQUET)
# failed = run_streaming_to_parquet(CSV_ROOT, CSV_PATTERN, OUTPUT_PARQUET)

# DuckDB quick check (after parquet exists):
# import duckdb
# duckdb.sql(f"SELECT * FROM '{OUTPUT_PARQUET}' LIMIT 10").show()